# Notebook 3 - Content-Based Filtering
**Content Based Filtering** is a recommendation technique that focuses on the properties of items themselves rather than user behavior. The core idea is that if a user enjoyed a particular movie, they are likely to enjoy other movies that share similar characteristics. Unlike collaborative filtering which asks "what do similar users like?", content based filtering asks "what is this item similar to?"

**TF-IDF** (Term Frequency-Inverse Document Frequency) converts movie genres into numerical vectors where each genre becomes a dimension. Cosine Similarity then measures the angle between two movie vectors to determine how similar they are — a score of 1.0 means identical genres, 0.0 means no genre overlap at all. Together these two techniques allow us to find movies that are most similar to ones a user has already enjoyed.

---

This notebook implements content based filtering on the MovieLens 1M dataset using movie genres as the primary feature. The key steps include:

- Loading and preprocessing movie genre data
- Cleaning genre formatting by replacing Sci-Fi with SciFi and Film-Noir with FilmNoir to ensure correct tokenization
- Building TF-IDF vectors for all 3,883 movies across 18 unique genres
- Computing a 3,883 × 3,883 cosine similarity matrix between all movies
- Building an item-to-item similarity function to find movies similar to a given title
- Generating personalized recommendations based on a user's highly rated movie history
- Saving the TF-IDF vectorizer, cosine similarity matrix and movie indices for use in later phases

---

**Key Limitation:** Genre only features mean movies with identical genre combinations receive the same similarity score and cannot be differentiated further. This limitation will be addressed in the hybrid model phase where content based scores are combined with SVD's personalized predictions.

In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
# load data
base_path = "OneDrive/Documents/Recommendation System"

movies = pd.read_csv(f'{base_path}/movies.csv')
train = pd.read_csv(f'{base_path}/train.csv')
test = pd.read_csv(f'{base_path}/test.csv')

print("Movies Shape:", movies.shape)
print(movies.head())

Movies Shape: (3883, 3)
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy


## Preprocess Genres

In [8]:
# clean the genre column by replacing \ with a space to make it readable for TF-IDF
# replace pipe separator with space
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# replace hyphens
movies['genres_clean'] = movies['genres_clean'].str.replace('Sci-Fi', 'SciFi', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace('Film-Noir', 'FilmNoir', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace("Children's", 'Childrens', regex=False)

# handle any missing value
movies['genres_clean'] = movies['genres_clean'].fillna('')

print("Sample genres before:", movies['genres'].iloc[0])
print("Sample genres after:", movies['genres_clean'].iloc[0])
print(movies[['title', 'genres', 'genres_clean']].head(10))

Sample genres before: Animation|Children's|Comedy
Sample genres after: Animation Childrens Comedy
                                title                        genres  \
0                    Toy Story (1995)   Animation|Children's|Comedy   
1                      Jumanji (1995)  Adventure|Children's|Fantasy   
2             Grumpier Old Men (1995)                Comedy|Romance   
3            Waiting to Exhale (1995)                  Comedy|Drama   
4  Father of the Bride Part II (1995)                        Comedy   
5                         Heat (1995)         Action|Crime|Thriller   
6                      Sabrina (1995)                Comedy|Romance   
7                 Tom and Huck (1995)          Adventure|Children's   
8                 Sudden Death (1995)                        Action   
9                    GoldenEye (1995)     Action|Adventure|Thriller   

                  genres_clean  
0   Animation Childrens Comedy  
1  Adventure Childrens Fantasy  
2               Comed

## Build TF_IDF Matrix

In [9]:
# convert all movies genres into a numerical matrix
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print("TF_IDF matrix shape:", tfidf_matrix.shape)
print("Number of unique genre terms:", len(tfidf.get_feature_names_out()))
print("Genre terms:", tfidf.get_feature_names_out())

TF_IDF matrix shape: (3883, 18)
Number of unique genre terms: 18
Genre terms: ['action' 'adventure' 'animation' 'childrens' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'filmnoir' 'horror' 'musical' 'mystery'
 'romance' 'scifi' 'thriller' 'war' 'western']


## Compute Cosine Similarity
A 3883×3883 matrix where cosine_sim[i][j] tells us how similar movie i is to movie j. For example:

- cosine_sim[0][0] = 1.0 (a movie is identical to itself)
- cosine_sim[0][5] = 0.8 (movies 0 and 5 share most genres)
- cosine_sim[0][100] = 0.0 (movies 0 and 100 share no genres)

In [10]:
# compute how similar every movie is to every other movie
# cosine similarity measures the angle between two vectors
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)
print("Sample similarity scores for movie 0:", cosine_sim[0][:10])

Cosine similarity matrix shape: (3883, 3883)
Sample similarity scores for movie 0: [1.         0.30552517 0.19737232 0.26019351 0.34435072 0.
 0.19737232 0.42515343 0.         0.        ]


## Create Movie Index Mapping

In [11]:
# creating two lookup dictionaries
# 1 for mapping movie titles to their row index
# and 2 for mapping movie id to their row index
movie_indices = pd.Series(
    movies.index,
    index=movies['title']
).drop_duplicates()

movie_id_to_idx = pd.Series(
    movies.index,
    index=movies['movie_id']
).drop_duplicates()

print("Total movies indexed:", len(movie_indices))
print("Sample Mapping:")
print(movie_indices.head())

Total movies indexed: 3883
Sample Mapping:
title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64


## Content Based Recommendation Function (by Title)

In [13]:
# creates a fuction that takes a movie title and returns the top N most similar movies
def get_similar_movies(title, cosine_sim=cosine_sim, movies_df=movies, n=10):
    
    # Check if the movie title exists in our dataset
    # If not, print a message and return None to avoid errors
    if title not in movie_indices:
        print(f"Movie '{title}' not found in dataset.")
        return None
    
    # Get the row index of this movie in the cosine similarity matrix
    # e.g. "Schindler's List (1993)" might be at index 522
    idx = movie_indices[title]
    
    # Get similarity scores of this movie with ALL other movies
    # enumerate adds the index so we know which movie each score belongs to
    # Result looks like: [(0, 0.3), (1, 0.8), (2, 0.0), ...]
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort movies from most similar to least similar based on score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the input movie itself by filtering on index instead of just skipping [0]
    sim_scores = [(i, score) for i, score in sim_scores 
                  if movies_df.iloc[i]['title'] != title]
        
    # Take top n
    sim_scores = sim_scores[:n]
    
    # Extract movie indices and similarity scores
    movie_idx = [i[0] for i in sim_scores]
    similarity = [round(i[1], 4) for i in sim_scores]
    
    # Build result dataframe
    result = movies_df.iloc[movie_idx][['title', 'genres']].copy()
    result['similarity_score'] = similarity
    
    return result.reset_index(drop=True)

print("Movies similar to Schindler's List (1993):")
print(get_similar_movies("Schindler's List (1993)"))

Movies similar to Schindler's List (1993):
                       title     genres  similarity_score
0         Richard III (1995)  Drama|War               1.0
1      Beyond Rangoon (1995)  Drama|War               1.0
2   Walking Dead, The (1995)  Drama|War               1.0
3  Courage Under Fire (1996)  Drama|War               1.0
4    Nothing Personal (1995)  Drama|War               1.0
5     Michael Collins (1996)  Drama|War               1.0
6             Platoon (1986)  Drama|War               1.0
7      Paths of Glory (1957)  Drama|War               1.0
8      Apocalypse Now (1979)  Drama|War               1.0
9                 Ran (1985)  Drama|War               1.0


## Testing with Multiple Movies

In [16]:
# testing similarity function with multiple movies
test_movies = [
    "Toy Story (1995)",
    "Terminator 2: Judgment Day (1991)",
    "Shawshank Redemption, The (1994)"
]

for movie in test_movies:
    print(f"\n--- Similar to: {movie} ---")
    print(get_similar_movies(movie, n=5))


--- Similar to: Toy Story (1995) ---
                                        title                       genres  \
0      Aladdin and the King of Thieves (1996)  Animation|Children's|Comedy   
1                    American Tail, An (1986)  Animation|Children's|Comedy   
2  American Tail: Fievel Goes West, An (1991)  Animation|Children's|Comedy   
3                   Rugrats Movie, The (1998)  Animation|Children's|Comedy   
4                        Bug's Life, A (1998)  Animation|Children's|Comedy   

   similarity_score  
0               1.0  
1               1.0  
2               1.0  
3               1.0  
4               1.0  

--- Similar to: Terminator 2: Judgment Day (1991) ---
                       title                  genres  similarity_score
0     Johnny Mnemonic (1995)  Action|Sci-Fi|Thriller               1.0
1   Nemesis 2: Nebula (1995)  Action|Sci-Fi|Thriller               1.0
2                Solo (1996)  Action|Sci-Fi|Thriller               1.0
3        Arrival, The 

# Personalized Recommendation Based on User History

In [18]:
def get_content_based_recommendations(user_id, train_df, movies_df, 
                                       cosine_sim=cosine_sim, n=10):
    
    # Filter to only movies this user rated 4 or higher
    user_ratings = train_df[
        (train_df['user_id'] == user_id) & 
        (train_df['rating'] >= 4)
    ].copy()
    
    # If user has no high ratings in train set, we can't make recommendations
    if user_ratings.empty:
        print(f"No high ratings found for user {user_id}")
        return None
    
    # Merge with movies dataframe to get title and genres
    # Use movie_id as the key to join both dataframes
    user_ratings = pd.merge(
        user_ratings[['user_id', 'movie_id', 'rating']],
        movies_df[['movie_id', 'title', 'genres']],
        on='movie_id',
        how='inner'
    )
    
    # Check if merge worked
    if 'title' not in user_ratings.columns or user_ratings.empty:
        print(f"Could not merge movie titles for user {user_id}")
        return None
    
    # Get ALL movies this user has rated for filtering later
    rated_movie_ids = set(train_df[train_df['user_id'] == user_id]['movie_id'].values)
    
    # Initialize empty score array with one slot per movie
    scores = np.zeros(len(movies_df))
    
    # Loop through each movie the user rated highly
    for _, row in user_ratings.iterrows():
        title = row['title']
        
        # Check if this movie exists in our index mapping
        if title in movie_indices:
            idx = movie_indices[title]
            scores += cosine_sim[idx]
    
    # Add accumulated scores to movies dataframe
    movies_copy = movies_df.copy()
    movies_copy['score'] = scores
    
    # Remove movies the user has already seen
    recommendations = movies_copy[~movies_copy['movie_id'].isin(rated_movie_ids)]
    
    # Sort by score and return top n
    recommendations = recommendations.sort_values('score', ascending=False).head(n)
    
    return recommendations[['title', 'genres', 'score']].reset_index(drop=True)

# Find most active user in train set
active_users = train['user_id'].value_counts()
test_user = active_users.index[0]

print(f"Content Based Recommendations for User {test_user}:")
print(get_content_based_recommendations(test_user, train, movies))

Content Based Recommendations for User 1680:
                                               title        genres       score
0                           Waiting to Exhale (1995)  Comedy|Drama  382.137436
1                         Eat Drink Man Woman (1994)  Comedy|Drama  382.137436
2                Guess Who's Coming to Dinner (1967)  Comedy|Drama  382.137436
3           Vie est belle, La (Life is Rosey) (1987)  Comedy|Drama  382.137436
4                                   Babyfever (1994)  Comedy|Drama  382.137436
5                    Midnight Dancers (Sibak) (1994)  Comedy|Drama  382.137436
6                                  Soft Fruit (1999)  Comedy|Drama  382.137436
7  Carriers Are Waiting, The (Les Convoyeurs Atte...  Comedy|Drama  382.137436
8                        Stefano Quantestorie (1993)  Comedy|Drama  382.137436
9                   Slaves to the Underground (1997)  Comedy|Drama  382.137436


## Save Model

In [19]:
import os

models_path = f'{base_path}/models'
os.makedirs(models_path, exist_ok=True)

with open(f'{models_path}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

np.save(f'{models_path}/cosine_sim.npy', cosine_sim)
movie_indices.to_pickle(f'{models_path}/movie_indices.pkl')

print("All content based models saved successfully!")

All content based models saved successfully!
